# Live Testing Notebook: RSFM Integration (`ML_LC_Classifier.rsfm`)

This notebook provides an end-to-end live testbench for the modular Remote Sensing Foundation Model (RSFM) subpackage. It exercises:
1. **Backbone Loading**: Toggle between **OpenGeoAI (`geoai-py`)** models on Colab/GPU (`USE_OPENGEOAI = True`) and the deterministic local CPU test backbone (`USE_OPENGEOAI = False`).
2. **Custom Embedding Stack (`extract_rsfm_embeddings`)**: Blended overlapping-chip extraction feeding into `load_and_split_training_data` + `tune_model`.
3. **Sparse-Polygon LoRA Fine-Tuning (`prepare_rsfm_chips` + `finetune_rsfm`)**: Freezing base weights, training LoRA adapters with `ignore_index=0`, and saving/reloading a portable checkpoint bundle.
4. **In-Memory Late Ancillary Fusion & Probability Streaming (`infer_rsfm_probabilities`)**: On-the-fly `WarpedVRT` alignment of DEM/proximity rasters + `hybrid_assembly.process_tile`.
5. **GEOBIA SNIC Segment Pooling (`pool_rsfm_by_segments` + `project_segment_probabilities`)**.
6. **Bi-Temporal Cosine Stability Mask (`compute_embedding_stability` + `filter_disturbed_samples` + `apply_transition_consistency`)**.

## 1. Environment Setup & Mode Toggle

In [ ]:
from pathlib import Path
import sys

project_root = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir()
)
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# --- CONFIGURATION TOGGLE ---
# Set USE_OPENGEOAI = True on Colab / GPU after running: pip install -e ".[rsfm]"
USE_OPENGEOAI = False
OPENGEOAI_MODEL_NAME = "prithvi_eo_v2_300"

# Optional: point to your real GeoTIFF/GeoJSON files, or leave None to generate a self-contained test dataset
CUSTOM_RASTER_T1 = None
CUSTOM_RASTER_T2 = None
CUSTOM_DEM = None
CUSTOM_SNIC_SEG = None
CUSTOM_SAMPLES = None

WORK_DIR = project_root / "output" / "rsfm_live_test"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working directory: {WORK_DIR}")

In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import box

from ML_LC_Classifier import (
    SimpleCPUBackbone,
    apply_transition_consistency,
    assign_segment_labels,
    bayesian_smooth,
    classify_from_probs,
    compute_embedding_stability,
    extract_rsfm_embeddings,
    filter_disturbed_samples,
    finetune_rsfm,
    infer_rsfm_probabilities,
    list_opengeoai_models,
    load_and_split_training_data,
    load_opengeoai_backbone,
    load_rsfm_checkpoint,
    pool_rsfm_by_segments,
    prepare_rsfm_chips,
    process_tile,
    project_segment_probabilities,
    tune_model,
)

## 2. Initialize Backbone (OpenGeoAI vs. Local CPU Test Backbone)

In [ ]:
if USE_OPENGEOAI:
    available = list_opengeoai_models()
    print(f"OpenGeoAI models found: {len(available)}")
    backbone = load_opengeoai_backbone(OPENGEOAI_MODEL_NAME)
    CHIP_SIZE = 224
    STRIDE = 112
else:
    backbone = SimpleCPUBackbone(embed_dim=16, seed=42)
    CHIP_SIZE = 16
    STRIDE = 8

print(f"Active backbone: {type(backbone).__name__} (embed_dim={backbone.embed_dim})")

## 3. Prepare Multi-Epoch Test Rasters, Coarse WGS84 DEM, SNIC Segments, & Sparse Polygons

In [ ]:
if all(p is not None for p in [CUSTOM_RASTER_T1, CUSTOM_RASTER_T2, CUSTOM_DEM, CUSTOM_SNIC_SEG, CUSTOM_SAMPLES]):
    s2_t1_path = Path(CUSTOM_RASTER_T1)
    s2_t2_path = Path(CUSTOM_RASTER_T2)
    dem_path = Path(CUSTOM_DEM)
    seg_path = Path(CUSTOM_SNIC_SEG)
    samples_path = Path(CUSTOM_SAMPLES)
else:
    # Generate a 48x48 10m UTM Zone 51S test scene (simulating Timor-Leste classes 1, 2, 7, 16)
    H, W, C = 48, 48, 4
    utm_transform = from_origin(780000.0, 9050000.0, 10.0, 10.0)
    rng = np.random.default_rng(42)

    s2_t1 = rng.normal(0.0, 0.005, size=(C, H, W)).astype(np.float32)
    # Quadrant 1 (top-left): Primary Forest (Class 1)
    s2_t1[:, :24, :24] += np.array([0.03, 0.05, 0.03, 0.42], dtype=np.float32)[:, None, None]
    # Quadrant 2 (top-right): Coffee Agroforestry (Class 7, similar NIR, slightly warmer SWIR/Red, high DEM)
    s2_t1[:, :24, 24:] += np.array([0.04, 0.06, 0.04, 0.39], dtype=np.float32)[:, None, None]
    # Quadrant 3 (bottom-left): Paddy Field (Class 11)
    s2_t1[:, 24:, :24] += np.array([0.06, 0.09, 0.07, 0.26], dtype=np.float32)[:, None, None]
    # Quadrant 4 (bottom-right): Built-up / Cleared (Class 16)
    s2_t1[:, 24:, 24:] += np.array([0.18, 0.21, 0.24, 0.16], dtype=np.float32)[:, None, None]

    # Epoch t2: bottom-left paddy converts to built-up clearing (real disturbance!)
    s2_t2 = s2_t1.copy()
    s2_t2[:, 24:, :24] = np.array([0.19, 0.22, 0.25, 0.15], dtype=np.float32)[:, None, None] + rng.normal(0.0, 0.005, size=(C, 24, 24)).astype(np.float32)

    s2_t1_path = WORK_DIR / "s2_2020.tif"
    s2_t2_path = WORK_DIR / "s2_2025.tif"
    for p, arr in [(s2_t1_path, s2_t1), (s2_t2_path, s2_t2)]:
        with rasterio.open(
            p, "w", driver="GTiff", height=H, width=W, count=C,
            dtype="float32", crs="EPSG:32751", transform=utm_transform, nodata=np.nan
        ) as dst:
            dst.write(arr)
            dst.descriptions = ("B02", "B03", "B04", "B08")

    # Coarser EPSG:4326 DEM (west low=120m, east high=1050m to separate Coffee >800m)
    dem_path = WORK_DIR / "nasadem_wgs84.tif"
    dem_arr = np.array([[[120.0, 120.0, 1050.0, 1050.0]] * 4], dtype=np.float32)
    with rasterio.open(
        dem_path, "w", driver="GTiff", height=4, width=4, count=1,
        dtype="float32", crs="EPSG:4326", transform=from_origin(125.50, -8.54, 0.02, 0.02), nodata=np.nan
    ) as dst:
        dst.write(dem_arr)
        dst.descriptions = ("NASADEM_HGT",)

    # Unified SNIC segments (including negative wrapped int32 GEE IDs)
    seg_path = WORK_DIR / "snic_segments.tif"
    seg_arr = np.zeros((1, H, W), dtype=np.int32)
    seg_arr[0, :24, :24] = 101
    seg_arr[0, :24, 24:] = 102
    seg_arr[0, 24:, :24] = -2000000001
    seg_arr[0, 24:, 24:] = -2000000002
    with rasterio.open(
        seg_path, "w", driver="GTiff", height=H, width=W, count=1,
        dtype="int32", crs="EPSG:32751", transform=utm_transform, nodata=0
    ) as dst:
        dst.write(seg_arr)

    # Sparse training polygons in EPSG:4326
    samples_path = WORK_DIR / "gt_samples.geojson"
    gdf = gpd.GeoDataFrame(
        {"class_id": [1, 7, 11, 16]},
        geometry=[
            box(780040.0, 9049820.0, 780160.0, 9049940.0),
            box(780280.0, 9049820.0, 780400.0, 9049940.0),
            box(780040.0, 9049580.0, 780160.0, 9049700.0),
            box(780280.0, 9049580.0, 780400.0, 9049700.0),
        ],
        crs="EPSG:32751",
    ).to_crs("EPSG:4326")
    gdf.to_file(samples_path, driver="GeoJSON")
    print("Synthetic Timor-Leste multi-epoch test fixtures created.")

## 4. Workflow 1: Windowed Embedding Stack -> `tune_model` (Shallow Baseline on RSFM Features)

In [ ]:
emb_result = extract_rsfm_embeddings(
    s2_t1_path,
    WORK_DIR / "embeddings_2020.tif",
    backbone=backbone,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    wavelengths_um=[0.49, 0.56, 0.66, 0.84],
)
print(f"Embedding bands: {len(emb_result.feature_names)} | File: {emb_result.stack_path.name}")

X_train, X_test, y_train, y_test = load_and_split_training_data(
    emb_result.stack_path,
    samples_path,
    class_field="class_id",
    test_size=0.3,
    random_state=42,
)
rf_tuned = tune_model(
    classifier="RandomForest",
    parameter_space={"n_estimators": [50]},
    X_train=X_train,
    y_train=y_train,
    cv=3,
    random_state=42,
)
rf_acc = float((rf_tuned.model.predict(X_test) == y_test).mean())
print(f"Held-out accuracy using RSFM embeddings + RandomForest: {rf_acc:.2%}")

## 5. Workflow 2: Sparse-Polygon LoRA Fine-Tuning & Checkpoint Round-Trip

In [ ]:
chip_dataset = prepare_rsfm_chips(
    s2_t1_path,
    samples_path,
    class_field="class_id",
    chip_size=CHIP_SIZE,
    wavelengths_um=[0.49, 0.56, 0.66, 0.84],
)
print(f"Extracted chips: {chip_dataset.chips.shape} | Classes: {chip_dataset.class_values}")

ft_result = finetune_rsfm(
    chip_dataset,
    backbone=backbone,
    lora_rank=4,
    lora_alpha=8.0,
    epochs=40,
    lr=0.15,
)
print(f"LoRA masked loss: {ft_result.loss_history[0]:.4f} -> {ft_result.loss_history[-1]:.4f}")

ckpt_dir = WORK_DIR / "lora_checkpoint"
ft_result.bundle.save(ckpt_dir)
bundle = load_rsfm_checkpoint(ckpt_dir, backbone=backbone)
print(f"Checkpoint saved & reloaded cleanly from {ckpt_dir}")

## 6. Workflow 3: In-Memory Late Ancillary Fusion -> Streaming `uint16` Probabilities -> `hybrid_assembly.process_tile`

In [ ]:
prob_path = infer_rsfm_probabilities(
    s2_t1_path,
    WORK_DIR / "rsfm_probs_2020_u16.tif",
    bundle=bundle,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    block_size=32,
    output_dtype="uint16",
)

hybrid_out = process_tile(
    prob_path=prob_path,
    seg_path=seg_path,
    out_dir=WORK_DIR / "hybrid_out",
    block_size=48,
    overlap=0,
)
print("Hybrid assembly deliverables:", {
    "lulc": str(hybrid_out.lulc.name),
    "stability": str(hybrid_out.stability.name),
    "ci": str(hybrid_out.ci.name),
})

## 7. Workflow 4: GEOBIA SNIC Segment Pooling (RSFM Embeddings + On-the-Fly `NASADEM`)

In [ ]:
seg_df = pool_rsfm_by_segments(
    s2_t1_path,
    seg_path,
    bundle=bundle,
    ancillary_paths=[dem_path],
    include_std=True,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    block_size=32,
)
labels_df = assign_segment_labels(seg_path, samples_path, class_field="class_id", block_size=32)
print(seg_df[["segment_id", "pixel_count", "embed_000_mean", "NASADEM_HGT_mean"]])
print("Segment labels:\n", labels_df)

## 8. Workflow 5: Bi-Temporal Embedding Cosine Stability Mask & Sample Disturbance Filter

In [ ]:
dist_path, mask_path = compute_embedding_stability(
    s2_t1_path,
    s2_t2_path,
    WORK_DIR / "cosine_dist_2020_2025.tif",
    WORK_DIR / "stable_mask_2020_2025.tif",
    bundle=bundle,
    tau_stable=0.15,
    chip_size=CHIP_SIZE,
    stride=STRIDE,
    block_size=32,
)

clean_samples, flagged_samples = filter_disturbed_samples(
    samples_path,
    dist_path,
    max_change_distance=0.15,
)
print("Sample stability flags:\n", flagged_samples[["class_id", "change_distance", "is_disturbed"]])

## 9. Optional Visual Inspection

In [ ]:
try:
    import matplotlib.pyplot as plt

    with rasterio.open(hybrid_out["lulc"]) as src_lc, \
         rasterio.open(dist_path) as src_dist, \
         rasterio.open(mask_path) as src_mask:
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].imshow(src_lc.read(1), cmap="tab20")
        axes[0].set_title("Hybrid RSFM + OBIA LULC (2020)")
        im1 = axes[1].imshow(src_dist.read(1), cmap="inferno", vmin=0, vmax=1.0)
        axes[1].set_title("Embedding Cosine Distance (2020->2025)")
        fig.colorbar(im1, ax=axes[1], fraction=0.046)
        axes[2].imshow(src_mask.read(1), cmap="gray", vmin=0, vmax=1)
        axes[2].set_title("Binary Stability Mask (1=Stable)")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib not installed; skipping visualization.")